# Cravveo Company — 파인튜닝 노트북

**프로젝트**: 100일 AI 파인튜닝 도전기  
**환경**: Google Colab T4 GPU  
**모델**: unsloth/gemma-2b-it

---

## ⚠️ 매 세션 시작 규칙
1. **셀 1~6을 순서대로 실행**
2. **셀 6 실행 후 반드시 런타임 재시작** (상단 메뉴 → 런타임 → 런타임 재시작)
3. 재시작 후 **셀 7부터 이어서 실행**

---

## 🔧 SECTION 0 — 매 세션 시작마다 실행 (Run Every Session)

In [1]:
# 셀 1 — GPU 확인
!nvidia-smi

Mon Jun 22 10:09:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 셀 2 — Unsloth + torchvision 설치 (반드시 한 셀에 같이)
%%capture
!pip install unsloth
!pip install --upgrade "torchvision>=0.27.0"

In [3]:
# 셀 3 — Unsloth 설치 확인
import unsloth
print("Unsloth version:", unsloth.__version__)
print("✅ Unsloth 설치 완료")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0622 10:14:57.047000 1628 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0622 10:14:57.090000 1628 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_librar

🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


Unsloth version: 2026.6.8
✅ Unsloth 설치 완료


In [4]:
# 셀 4 — 나머지 라이브러리 설치
%%capture
!pip install transformers datasets accelerate peft trl bitsandbytes

In [5]:
# 셀 5 — huggingface_hub 설치
%%capture
!pip install -q huggingface_hub

In [6]:
# 셀 6 — 버전 확인
import torch
import transformers
import datasets
import peft
import trl
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("✅ 모든 라이브러리 확인 완료")

torch: 2.12.1+cu130
transformers: 5.5.0
datasets: 4.3.0
peft: 0.19.1
trl: 0.24.0
✅ 모든 라이브러리 확인 완료


## ⚠️ 여기서 런타임 재시작!

상단 메뉴 → **런타임** → **런타임 재시작**  
재시작 후 아래 셀 7부터 이어서 실행하세요.

---

## 🤖 SECTION 1 — Day 018: 모델 로드

In [1]:
# 셀 6.5 — bitsandbytes CUDA 수정
import subprocess, ctypes

result = subprocess.run(["find", "/usr", "-name", "libnvJitLink.so.13"], capture_output=True, text=True)
paths = result.stdout.strip().split("\n")

if paths and paths[0]:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"✅ CUDA 라이브러리 로드 완료: {paths[0]}")
else:
    print("⚠️ libnvJitLink.so.13을 찾지 못했습니다 — 그냥 다음 셀로 진행하세요")


✅ CUDA 라이브러리 로드 완료: /usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13


In [2]:
# 셀 7 — 모델 로드 (Llama 3.2 1B로 변경)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)
print("✅ 모델 로드 성공")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0622 10:16:38.008000 3655 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0622 10:16:38.039000 3655 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ 모델 로드 성공


In [3]:
# 셀 8 — LoRA 어댑터 설정
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ LoRA 어댑터 설정 완료")

Unsloth 2026.6.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


✅ LoRA 어댑터 설정 완료


In [ ]:
# 디버그 테스트 — 데이터 3개로 loss가 정상인지 확인  테스트라 실제는 건너뜀
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 초간단 데이터 3개
texts = [
    f"질문: 안녕?\n답변: 안녕하세요!{tokenizer.eos_token}",
    f"질문: 크라베오가 뭐야?\n답변: 1인 기업이야.{tokenizer.eos_token}",
    f"질문: 오늘 뭐 했어?\n답변: 파인튜닝 공부했어.{tokenizer.eos_token}",
]

# 토큰 수 확인
for i, t in enumerate(texts):
    tokens = tokenizer.encode(t)
    print(f"[{i}] {len(tokens)} tokens")

dataset_test = Dataset.from_dict({"text": texts})

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_test,
    dataset_text_field = "text",
    max_seq_length = 128,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 0,
        max_steps = 10,
        learning_rate = 2e-5,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 42,
    ),
)

trainer.train()


In [8]:
# 셀 8.5 - Day 027 — Before 테스트 → 학습 → After 테스트 (한방에)
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ========== 1. Before 테스트 ==========
FastLanguageModel.for_inference(model)

test_questions = [
    "질문: 크라베오 컴퍼니가 뭐야?\n답변:",
    "질문: 크라베오의 목표가 뭐야?\n답변:",
    "질문: Build in Public이 뭐야?\n답변:",
    "질문: 파인튜닝을 왜 배우고 있어?\n답변:",
]

print("=" * 50)
print("📋 BEFORE — 학습 전 모델 답변")
print("=" * 50)
before_answers = []
for q in test_questions:
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    before_answers.append(answer)
    print()
    print(answer)
    print("-" * 50)

# ========== 2. 데이터 로드 + 변환 ==========
FastLanguageModel.for_training(model)

dataset = load_dataset("json", data_files="cravveo_combined_dataset.jsonl", split="train")
print(f"\n✅ 데이터셋 로드: {len(dataset)}개")

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        text = f"질문: {instruction}\n답변: {output}{EOS_TOKEN}"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# ========== 3. 학습 ==========
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 30, #— 같은 데이터를 더 많이 반복
        learning_rate = 5e-5, #— 더 강하게 학습
        fp16 = True,
        logging_steps = 5,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 42,
        save_strategy = "no",
    ),
)

print("\n🚀 학습 시작!")
trainer_stats = trainer.train()
print(f"✅ 학습 완료! 최종 loss: {trainer_stats.metrics['train_loss']:.4f}")

# ========== 4. After 테스트 ==========
FastLanguageModel.for_inference(model)

print("\n" + "=" * 50)
print("📋 AFTER — 학습 후 모델 답변")
print("=" * 50)
for q in test_questions:
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print()
    print(answer)
    print("-" * 50)

# ========== 5. 비교 ==========
print("\n" + "=" * 50)
print("📊 Before vs After 비교")
print("=" * 50)
for i, q in enumerate(test_questions):
    question = q.split("\n")[0]
    print(f"\n{question}")
    print(f"  Before: {before_answers[i].split('답변:')[1][:80] if '답변:' in before_answers[i] else 'N/A'}")
    after_inputs = tokenizer(q, return_tensors="pt").to("cuda")
    print(f"  After:  (위 결과 참조)")
    print()


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📋 BEFORE — 학습 전 모델 답변


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



질문: 크라베오 컴퍼니가 뭐야?
답변: 크라베오 컴퍼니는 20세기 초에 미국에서 수학자에 의해 설립된 개인적 수학자와의 협력으로 인해 수학의 발전에 기여했다. 20세기 초에 크라베오가 수학의 개념을 정리한 seminal work인 "수학의 개념" (1921)가 있다. 크라베오가 수학의 개념을 정리한 seminal work인
--------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



질문: 크라베오의 목표가 뭐야?
답변: 크라베오의 목표는 "자연을 사용하여 유기체의 성장을 지원하고, 유기체의 발전과 지속을 위해 다양한 방식에 대해 연구하고, 유기체의 발전과 지속을 위해 새로운 기술과 방법을 개발하고, 유기체의 발전과 지속을 위해 유기체의 발전과 지속에 대한 새로운 연구를ดำเนินการ하는 것"이었다. 크라베오의 목표
--------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



질문: Build in Public이 뭐야?
답변: Build in Public이 뭐야?

Build in Public은,_build_in_public_(,r) Build_in_public( "build_in_public",r) "build_in_public"를 의미합니다.

Build_in_public는 "build_in_public"을 의미합니다.

Build_in_public은 "build_in_public"가란 지식공간을 의미합니다.

Build_in_public은 "build_in_public"가란 지식공간을 의미합니다.

Build_in_public은 "build_in_public"가
--------------------------------------------------

질문: 파인튜닝을 왜 배우고 있어?
답변: 파인튜팅은 재미있는 시간을 보내는 것을 통해 인생에 큰 기회를 제공할 수 있는 좋은 시작점을 제공합니다. 파인튜팅은 일상에서 일과를 더 깊게 이해하고, 다른 사람들과의 관계를 deepen하게 할 수 있는 좋은 기회를 제공합니다. 또한, 파인튜팅은 취미를 찾는 기회를 제공할 수 있습니다. 파인튜팅을 배우고 싶은 사람들은 모두
--------------------------------------------------


FileNotFoundError: Unable to find '/content/cravveo_combined_dataset.jsonl'

In [9]:
# 셀 9 — GPU 메모리 확인
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"전체 메모리: {max_memory} GB")
print(f"현재 사용 중: {start_gpu_memory} GB")
print(f"남은 메모리: {round(max_memory - start_gpu_memory, 3)} GB")

GPU: Tesla T4
전체 메모리: 14.563 GB
현재 사용 중: 2.346 GB
남은 메모리: 12.217 GB


## 📄 SECTION 2 — Day 019: JSONL 포맷 분석

In [10]:
# 셀 10 — JSONL 샘플 5개 작성 + 저장
import json

data = [
    {"instruction": "크라베오 회사를 소개해줘", "input": "", "output": "크라베오는 1인 기업 자동화를 연구하는 회사입니다. AI와 Python을 활용해 반복 업무를 줄이는 시스템을 만듭니다."},
    {"instruction": "파인튜닝이 뭔지 쉽게 설명해줘", "input": "", "output": "파인튜닝은 이미 만들어진 AI 모델을 내 데이터로 추가 학습시키는 과정입니다. AI가 내 스타일과 지식을 갖게 됩니다."},
    {"instruction": "Unsloth를 쓰는 이유가 뭐야?", "input": "", "output": "Unsloth는 파인튜닝 속도를 2~5배 빠르게 하고 메모리를 60% 절약합니다. 무료 코랩 환경에서 파인튜닝을 현실적으로 가능하게 해주는 도구입니다."},
    {"instruction": "LoRA가 뭐야?", "input": "", "output": "LoRA는 모델 전체를 학습시키지 않고 일부 층에만 작은 어댑터를 붙여서 학습하는 방법입니다. 메모리와 시간을 크게 절약할 수 있습니다."},
    {"instruction": "JSONL 포맷이 뭐야?", "input": "", "output": "JSONL은 JSON 데이터를 한 줄에 하나씩 나열한 파일 형식입니다. 파인튜닝 학습 데이터를 저장하는 데 표준으로 쓰입니다."},
]

with open("cravveo_sample.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ cravveo_sample.jsonl 생성 완료 — {len(data)}개")

✅ cravveo_sample.jsonl 생성 완료 — 5개


In [11]:
# 셀 11 — JSONL 파일 읽기 확인
with open("cravveo_sample.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"총 {len(lines)}줄\n")
for i, line in enumerate(lines):
    item = json.loads(line)
    print(f"[{i+1}] Q: {item['instruction']}")
    print(f"     A: {item['output'][:40]}...")
    print()

총 5줄

[1] Q: 크라베오 회사를 소개해줘
     A: 크라베오는 1인 기업 자동화를 연구하는 회사입니다. AI와 Python을...

[2] Q: 파인튜닝이 뭔지 쉽게 설명해줘
     A: 파인튜닝은 이미 만들어진 AI 모델을 내 데이터로 추가 학습시키는 과정입...

[3] Q: Unsloth를 쓰는 이유가 뭐야?
     A: Unsloth는 파인튜닝 속도를 2~5배 빠르게 하고 메모리를 60% 절...

[4] Q: LoRA가 뭐야?
     A: LoRA는 모델 전체를 학습시키지 않고 일부 층에만 작은 어댑터를 붙여서...

[5] Q: JSONL 포맷이 뭐야?
     A: JSONL은 JSON 데이터를 한 줄에 하나씩 나열한 파일 형식입니다. ...



In [12]:
# 셀 12 — HuggingFace datasets 라이브러리로 로드
from datasets import load_dataset

dataset = load_dataset("json", data_files="cravveo_sample.jsonl", split="train")
print(f"데이터셋 크기: {len(dataset)}개")
print(f"컬럼: {dataset.column_names}")
print()
for i in range(len(dataset)):
    print(f"[{i+1}] {dataset[i]['instruction']}")

Generating train split: 0 examples [00:00, ? examples/s]

데이터셋 크기: 5개
컬럼: ['instruction', 'input', 'output']

[1] 크라베오 회사를 소개해줘
[2] 파인튜닝이 뭔지 쉽게 설명해줘
[3] Unsloth를 쓰는 이유가 뭐야?
[4] LoRA가 뭐야?
[5] JSONL 포맷이 뭐야?


## 📝 SECTION 3 — Day 020+021: 크라베오 데이터셋 제작

In [13]:
# 셀 13 — 크라베오 정체성 데이터 10개 작성 + 저장
import json

cravveo_data = [
    {"instruction": "크라베오 컴퍼니가 뭐야?", "input": "", "output": "IT 프로그램 개발, 인터넷 쇼핑몰, 유튜브 채널 등을 혼자서 운영하면서 수익을 창출하는 1인 기업이야."},
    {"instruction": "왜 1인 기업을 선택했어?", "input": "", "output": "회사 생활을 20년 넘게 하면서 이리 치이고 저리 치이다 보니까 혼자 하는 게 낫겠다 싶었어. 그래서 시작했어."},
    {"instruction": "100일 AI 도전이 뭐야?", "input": "", "output": "지금 내가 하고 있는 것들을 홍보도 하고 기록도 남기고 싶어서 도전하고 있어."},
    {"instruction": "파인튜닝을 왜 배우고 있어?", "input": "", "output": "결국 개인의 지식만이 살아남는다는 걸 배우면서 느꼈어. 클라우드 AI 비용 문제도 있고, 나만의 AI를 만들면 파인튜닝으로 수익도 낼 수 있지 않을까 싶어서 배우고 있어."},
    {"instruction": "크라베오의 목표가 뭐야?", "input": "", "output": "1인 스타트업으로 살아남는 거야. 그게 지금 목표야."},
    {"instruction": "유튜브를 왜 시작했어?", "input": "", "output": "홍보랑 기록용이야. 내가 뭘 하고 있는지 남기고 싶었어."},
    {"instruction": "하루를 어떻게 보내?", "input": "", "output": "아침에 생산직 회사에 출근해서 일하고, 오후 6시쯤 퇴근하면 1인 스타트업 만들려고 공부하고 유튜브 영상 만들어. 저녁 12시쯤에 집에 가서 새벽 1~3시쯤에 자."},
    {"instruction": "AI를 처음 배울 때 가장 어려웠던 게 뭐야?", "input": "", "output": "시작하는 거야. 컴퓨터 켜고 도전해보는 게 제일 어려웠어. 이걸 왜 해야 하는지도 몰랐으니까."},
    {"instruction": "Build in Public이 뭐야?", "input": "", "output": "배우는 과정을 숨기지 않고 공개하면서 진행하는 방식이야. 완성된 결과만 보여주는 게 아니라 실패도 에러도 다 공개하는 거야. 내가 100일 도전을 유튜브에 올리는 것도 Build in Public이야."},
    {"instruction": "크라베오 AI가 완성되면 뭘 할 거야?", "input": "", "output": "당연히 수익을 창출해야지. 그게 목적이야."},
]

with open("cravveo_identity.jsonl", "w", encoding="utf-8") as f:
    for item in cravveo_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ cravveo_identity.jsonl 저장 완료 — {len(cravveo_data)}개")

✅ cravveo_identity.jsonl 저장 완료 — 10개


In [14]:
# 셀 14 — JSONL 형식 검증
print("--- JSONL 검증 ---\n")
errors = 0
with open("cravveo_identity.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            item = json.loads(line)
            print(f"[{i+1}] ✅ {item['instruction'][:30]}")
        except json.JSONDecodeError as e:
            print(f"[{i+1}] ❌ 형식 오류: {e}")
            errors += 1

print(f"\n총 오류: {errors}개" if errors else "\n✅ 모든 항목 형식 정상")

--- JSONL 검증 ---

[1] ✅ 크라베오 컴퍼니가 뭐야?
[2] ✅ 왜 1인 기업을 선택했어?
[3] ✅ 100일 AI 도전이 뭐야?
[4] ✅ 파인튜닝을 왜 배우고 있어?
[5] ✅ 크라베오의 목표가 뭐야?
[6] ✅ 유튜브를 왜 시작했어?
[7] ✅ 하루를 어떻게 보내?
[8] ✅ AI를 처음 배울 때 가장 어려웠던 게 뭐야?
[9] ✅ Build in Public이 뭐야?
[10] ✅ 크라베오 AI가 완성되면 뭘 할 거야?

✅ 모든 항목 형식 정상


In [15]:
# 셀 15 — 전체 데이터셋 합치기
# cravveo_briefing_dataset.jsonl 파일을 미리 업로드해두세요
# 코랩 왼쪽 파일 탭 → 업로드 버튼

source_files = [
    "cravveo_briefing_dataset.jsonl",
    "cravveo_identity.jsonl",
]

all_data = []
for fname in source_files:
    try:
        with open(fname, "r", encoding="utf-8") as f:
            lines = f.readlines()
            all_data.extend(lines)
            print(f"✅ {fname} — {len(lines)}개 로드")
    except FileNotFoundError:
        print(f"⚠️ {fname} 없음 — 건너뜀")

with open("cravveo_full_dataset.jsonl", "w", encoding="utf-8") as f:
    f.writelines(all_data)

print(f"\n총 {len(all_data)}개 → cravveo_full_dataset.jsonl 완성")

⚠️ cravveo_briefing_dataset.jsonl 없음 — 건너뜀
✅ cravveo_identity.jsonl — 10개 로드

총 10개 → cravveo_full_dataset.jsonl 완성


In [16]:
# 셀 16 — 전체 데이터셋 샘플 확인
with open("cravveo_full_dataset.jsonl", "r", encoding="utf-8") as f:
    all_lines = f.readlines()

total = len(all_lines)
print(f"전체 데이터: {total}개\n")

print("=== 처음 3개 ===")
for line in all_lines[:3]:
    item = json.loads(line)
    print(f"Q: {item['instruction']}")
    print(f"A: {item['output'][:50]}...")
    print()

print("=== 마지막 3개 ===")
for line in all_lines[-3:]:
    item = json.loads(line)
    print(f"Q: {item['instruction']}")
    print(f"A: {item['output'][:50]}...")
    print()

전체 데이터: 10개

=== 처음 3개 ===
Q: 크라베오 컴퍼니가 뭐야?
A: IT 프로그램 개발, 인터넷 쇼핑몰, 유튜브 채널 등을 혼자서 운영하면서 수익을 창출하는 ...

Q: 왜 1인 기업을 선택했어?
A: 회사 생활을 20년 넘게 하면서 이리 치이고 저리 치이다 보니까 혼자 하는 게 낫겠다 싶었...

Q: 100일 AI 도전이 뭐야?
A: 지금 내가 하고 있는 것들을 홍보도 하고 기록도 남기고 싶어서 도전하고 있어....

=== 마지막 3개 ===
Q: AI를 처음 배울 때 가장 어려웠던 게 뭐야?
A: 시작하는 거야. 컴퓨터 켜고 도전해보는 게 제일 어려웠어. 이걸 왜 해야 하는지도 몰랐으니...

Q: Build in Public이 뭐야?
A: 배우는 과정을 숨기지 않고 공개하면서 진행하는 방식이야. 완성된 결과만 보여주는 게 아니라...

Q: 크라베오 AI가 완성되면 뭘 할 거야?
A: 당연히 수익을 창출해야지. 그게 목적이야....



## 🤗 SECTION 4 — Day 022: HuggingFace 로그인

In [17]:
# 셀 17 — HuggingFace 로그인 (Colab Secrets 방식)
# 사전 준비: 코랩 왼쪽 🔑 아이콘 → + 새 비밀 추가
#   이름: HF_TOKEN
#   값: 허깅페이스 토큰 붙여넣기
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("✅ HuggingFace 로그인 완료")

✅ HuggingFace 로그인 완료


In [18]:
# 셀 18 — 로그인 확인
from huggingface_hub import whoami

user_info = whoami()
print(f"✅ 로그인 성공!")
print(f"사용자명: {user_info['name']}")

✅ 로그인 성공!
사용자명: cravveo


In [19]:
# 셀 18.5 - HuggingFace에 확장 데이터셋 업로드
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="cravveo_combined_dataset.jsonl",
    path_in_repo="cravveo_combined_dataset.jsonl",
    repo_id="cravveo/cravveo-briefing-dataset",
    repo_type="dataset",
)
print("✅ HuggingFace 업로드 완료 — 83개 데이터")


ValueError: Provided path: 'cravveo_combined_dataset.jsonl' is not a file on the local file system

## 📥 SECTION 5 — Day 024: HuggingFace 데이터셋 연결

In [ ]:
# 셀 19 — HuggingFace에서 내 데이터셋 불러오기
from datasets import load_dataset

dataset = load_dataset("cravveo/cravveo-briefing-dataset", split="train")

print(f"✅ 데이터셋 로드 완료")
print(f"데이터 수: {len(dataset)}개")
print(f"컬럼: {dataset.column_names}")
print()
print("--- 첫 번째 데이터 미리보기 ---")
print("instruction:", dataset[0]['instruction'][:80])
print("output:", dataset[0]['output'][:80])

## 🔄 SECTION 6 — Day 026: 프롬프트 템플릿 수정 + 학습 재실행

Day 025에서 loss가 25.5에서 시작한 원인:  
`### 질문: / ### 답변:` 마크다운 형식은 Gemma가 모르는 포맷이었습니다.

**수정**: Gemma의 채팅 템플릿 `<start_of_turn>` 형식으로 교체합니다.

---

In [ ]:
# 셀 20 — 프롬프트 변환 (Llama 3.2용)
dataset_clean = dataset.filter(lambda x: len(x["output"]) <= 300)
print(f"원본: {len(dataset)}개 → 필터링 후: {len(dataset_clean)}개")

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        text = f"질문: {instruction}\n답변: {output}{EOS_TOKEN}"
        texts.append(text)
    return {"text": texts}

dataset = dataset_clean.map(formatting_prompts_func, batched=True)

print("✅ 변환 완료")
print(dataset[0]['text'][:300])


In [ ]:
# 셀 21 — SFTTrainer 설정 (Llama 3.2 + 저장 에러 수정)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        num_train_epochs = 3,
        learning_rate = 2e-5,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 42,
        save_strategy = "no",
    ),
)

print("✅ SFTTrainer 설정 완료")


In [ ]:
# 셀 22 — 학습 실행
print("🚀 학습 시작!")
trainer_stats = trainer.train()
print("✅ 학습 완료!")
print(f"총 학습 시간: {trainer_stats.metrics['train_runtime']:.1f}초")
print(f"최종 loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# 셀 23 — Day 025 vs Day 026 loss 비교
print("=" * 50)
print("📊 Day 025 vs Day 026 비교")
print("=" * 50)
print()
print("Day 025 (### 질문/답변 형식):")
print("  Step 1 loss:  25.5")
print("  Step 30 loss: ~15.4")
print()
print("Day 026 (Gemma 채팅 템플릿):")
print(f"  최종 loss: {trainer_stats.metrics['train_loss']:.4f}")
print()
if trainer_stats.metrics['train_loss'] < 5:
    print("✅ 프롬프트 템플릿 수정 성공! loss가 정상 범위입니다.")
else:
    print("⚠️ loss가 아직 높습니다. 데이터 확인이 필요합니다.")

In [ ]:
# 셀 24 — 학습 전 vs 학습 후 비교 테스트
FastLanguageModel.for_inference(model)

test_questions = [
    "질문: 크라베오 컴퍼니가 뭐야?\n답변:",
    "질문: 크라베오의 목표가 뭐야?\n답변:",
    "질문: Build in Public이 뭐야?\n답변:",
]

print("=" * 50)
print("🤖 파인튜닝된 모델의 답변")
print("=" * 50)

for q in test_questions:
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print()
    print(answer)
    print("-" * 50)


In [4]:
# Step 2 — LoRA 가중치 저장
model.save_pretrained("cravveo-lora-adapter")
tokenizer.save_pretrained("cravveo-lora-adapter")
print("✅ LoRA 가중치 저장 완료")

import os
for f in os.listdir("cravveo-lora-adapter"):
    size = os.path.getsize(f"cravveo-lora-adapter/{f}") / 1024
    print(f"  {f} ({size:.1f} KB)")


Unsloth: Restored added_tokens_decoder metadata in cravveo-lora-adapter/tokenizer_config.json.


✅ LoRA 가중치 저장 완료
  chat_template.jinja (3.7 KB)
  tokenizer_config.json (49.5 KB)
  README.md (5.1 KB)
  tokenizer.json (16806.6 KB)
  adapter_config.json (1.2 KB)
  adapter_model.safetensors (44061.0 KB)


In [5]:
# Step 3 — HuggingFace에 모델 업로드
from google.colab import userdata

model.push_to_hub("cravveo/cravveo-llama-lora", token=userdata.get('HF_TOKEN'))
tokenizer.push_to_hub("cravveo/cravveo-llama-lora", token=userdata.get('HF_TOKEN'))
print("✅ HuggingFace 업로드 완료 — cravveo/cravveo-llama-lora")


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.3kB / 45.1MB            

Saved model to https://huggingface.co/cravveo/cravveo-llama-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpg3mzquis/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpg3mzquis/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

✅ HuggingFace 업로드 완료 — cravveo/cravveo-llama-lora


In [6]:
# Step 4 — 저장된 모델 다시 불러오기 테스트
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "cravveo/cravveo-llama-lora",
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_questions = [
    "질문: 크라베오 컴퍼니가 뭐야?\n답변:",
    "질문: 크라베오의 목표가 뭐야?\n답변:",
    "질문: Build in Public이 뭐야?\n답변:",
]

print("✅ HuggingFace에서 모델 로드 완료")
print("=" * 50)
for q in test_questions:
    inputs = tokenizer(q, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print()
    print(answer)
    print("-" * 50)


==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ HuggingFace에서 모델 로드 완료


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


질문: 크라베오 컴퍼니가 뭐야?
답변: 크라베오 컴퍼니는 유대인 종목, specifically 3% 이하의 종목에 대해 주식의 투자나 금융 시장에서 상당한 수익을 얻는 것을 목표로 하는 주식 재단이자 유도하는 주식 재단을 운영하는 무역 및 유tick시 회사이다. 

다음은 이 회사에 대한 더 cụ thể한 정보입니다.

*   3% 이하의 종목에 대해
--------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



질문: 크라베오의 목표가 뭐야?
답변: 크라베오의 목표는 전제가의 성과를 달성하여 자신을 재지시하는 것이다. 크라베오는 자신의 목표를 달성할 수 있는 methods을 찾아서 목표를 달성할 수 있도록 도움을 주는 것이 크라베오의 목표이다. 크라베오의 목표는 자신의 목표를 달성할 수 있는 methods을 찾아서 목표를 달성할 수 있도록 도움을 주
--------------------------------------------------

질문: Build in Public이 뭐야?
답변: Build in Public은 사람들의 삶을 개선하고자 하는 것이기에, Build in Public은 사람들의 삶을 개선하고자 하는 것을 의미한다. Build in Public은 사람들의 삶을 개선하고자 하는 것이지, Build in Public은 사람들의 삶을 개선하고자 하는 것을 의미한다. Build in Public은 사람들의 삶을 개선하고자 하는 것이지, Build in Public은 사람들의 삶을 개선하고자 하는 것을 의미한다. Build
--------------------------------------------------
